# EFO Transcript Clustering Pipeline
**Goal:** Identify the most common concerns of ALICE vs. Above ALICE community members from focus group transcripts.

**Pipeline overview:**
1. Load & parse transcripts → speaker-level rows (role = unknown)
2. LLM: role detection — facilitator vs. participant (facilitators removed)
3. LLM: 3-vote ALICE classification per participant speaker
4. Human review cell for classification disagreements
5. LLM: concern extraction per speaker
6. Explode concerns → one row per concern
7. Filter to ALICE speakers
8. Embed concerns with nomic-embed-text
9. UMAP dimensionality reduction
10. HDBSCAN clustering
11. LLM-named clusters + word cloud sanity check
12. 2D UMAP visualization
13. Save ALICE outputs
14. Run Above ALICE through the same pipeline
15. Side-by-side comparison: ALICE vs. Above ALICE
16. Side-by-side 2D UMAP visualization

## 0. Imports & Configuration

In [4]:
import os
import re
import json
import time
import ollama
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from collections import Counter
from wordcloud import WordCloud
import umap
import hdbscan


# ── Configuration ─────────────────────────────────────────────────────────────
TRANSCRIPTS_DIR     = "../data"       # folder containing all .txt files
OLLAMA_MODEL        = "qwen2.5:14b"        # classification + concern extraction + cluster naming
EMBED_MODEL         = "nomic-embed-text"   # embeddings

# Classification
N_VOTES             = 3                    # number of classification runs per speaker

# UMAP
UMAP_N_COMPONENTS   = 5                    # dimensions for HDBSCAN input
UMAP_N_COMPONENTS_2D = 2                   # dimensions for visualization
UMAP_METRIC         = "cosine"
UMAP_RANDOM_STATE   = 42

# HDBSCAN — keep small given ~150-300 row dataset
HDBSCAN_MIN_CLUSTER_SIZE = 3
HDBSCAN_MIN_SAMPLES      = 2

print("Configuration loaded.")


# ── Shared LLM utility — available to all cells ───────────────────────────────
def call_ollama_with_retry(prompt, temperature, retries=3):
    """Call Ollama chat with exponential backoff on failure.

    Defined here in Cell 0 so it is available to role detection (Cell 2),
    classification (Cell 3), and concern extraction (Cell 4) without
    any cell needing to import or redefine it.

    Returns the raw JSON string, or None if all retries fail.
    """
    for attempt in range(retries):
        try:
            response = ollama.chat(
                model=OLLAMA_MODEL,
                messages=[{"role": "user", "content": prompt}],
                format="json",
                options={"temperature": temperature},
            )
            raw = response["message"]["content"].strip()
            raw = re.sub(r"^```json\s*|^```\s*|\s*```$", "", raw, flags=re.MULTILINE).strip()
            return raw
        except Exception as e:
            wait = 2 ** attempt   # 1s, 2s, 4s
            print(f"  Ollama call failed (attempt {attempt + 1}/{retries}): {e} — retrying in {wait}s")
            time.sleep(wait)
    print("  All retries exhausted — returning None")
    return None

Configuration loaded.


## 1. Parse Transcripts → Speaker-Level Dataframe
Each row = one speaker in one transcript.  
Transcripts without `Speaker N` markers are skipped.

In [5]:
def parse_header(text):
    """Extract metadata from the top section of a transcript."""
    header = {}
    for line in text.split("\n")[:25]:
        if ":" in line:
            key, _, val = line.partition(":")
            header[key.strip()] = val.strip()
    return header


def extract_population_tags(header):
    """Pull actual demographic tags from the Population line.
    
    The Population line is a template — actual tags appear after 'White/BIPOC,'
    """
    pop_line = header.get(
        "Population - identify one or more of the following groups", ""
    )
    parts = pop_line.split(",")
    actual_tags, found_bipoc = [], False
    for p in parts:
        if "white/bipoc" in p.lower():
            found_bipoc = True
            continue
        if found_bipoc:
            actual_tags.append(p.strip())
    return ", ".join(actual_tags) if actual_tags else pop_line


def parse_speakers(text):
    """Split transcript body into per-speaker utterance lists.
    
    Returns dict {speaker_id: [utterance, ...]} or None if unlabeled.
    """
    # Match only to end of the SUMMARY KEYWORDS line — no re.DOTALL,
    # which would over-consume across newlines and skip the transcript body.
    body_match = re.search(r"SUMMARY KEYWORDS[^\n]*\n", text)
    body = text[body_match.end():] if body_match else text

    speaker_pattern = re.compile(r"^(Speaker \d+)\s*\n", re.MULTILINE)
    matches = list(speaker_pattern.finditer(body))

    if not matches:
        return None  # unlabeled transcript — skip

    speakers = {}
    for i, m in enumerate(matches):
        speaker = m.group(1)
        start   = m.end()
        end     = matches[i + 1].start() if i + 1 < len(matches) else len(body)
        utterance = body[start:end].strip()
        if utterance:
            speakers.setdefault(speaker, []).append(utterance)
    return speakers


def build_speaker_dataframe(transcripts_dir):
    """Load all transcripts and build one row per speaker.
    
    Speaker 1 is flagged as facilitator. Unlabeled transcripts are skipped.
    """
    rows, skipped = [], []

    for filepath in sorted(Path(transcripts_dir).glob("*.txt")):
        text     = filepath.read_text(encoding="utf-8", errors="ignore")
        header   = parse_header(text)
        speakers = parse_speakers(text)

        if speakers is None:
            skipped.append(filepath.name)
            continue

        pop_tags = extract_population_tags(header)

        for speaker_id, utterances in speakers.items():
            full_text = " ".join(utterances).strip()
            if not full_text:
                continue
            rows.append({
                "filename":        filepath.name,
                "date":            header.get("Date of conversation", ""),
                "location":        header.get("Meeting location", ""),
                "facilitator":     header.get("Name of Facilitator(s)", ""),
                "attendees":       header.get("Number of attendees", ""),
                "population_tags": pop_tags,
                "speaker_id":      speaker_id,
                "speaker_role":    "unknown",   # determined by LLM in Cell 2
                "full_text":       full_text,
            })

    df = pd.DataFrame(rows)
    print(f"Loaded {df['filename'].nunique()} transcripts → {len(df)} speaker rows")
    print(f"Skipped {len(skipped)} unlabeled transcripts: {skipped}")
    return df


df_speakers = build_speaker_dataframe(TRANSCRIPTS_DIR)
df_speakers.head()

Loaded 23 transcripts → 185 speaker rows
Skipped 8 unlabeled transcripts: ['EFO_75 State Street__Cumberland County, Older Adults, Urban, Above ALICE.txt', 'EFO_DECDSubcommittee_Both Counties, Urban, Above ALICE.txt', 'EFO_Investment Vol_Cumberland County, Urban, Above ALICE, UWSM Volunteers.txt', 'EFO_Loaned Execs_Cumberland County, Corporate Employees, Above ALICE, UWSM Volunteers.txt', 'EFO_Noble Students and Ambassadors 1_York, Rural, Under 25, Most ALICE.txt', 'EFO_Noble Students and Ambassadors 2_York, Rural, Under 25, Most ALICE.txt', 'EFO_Noble Teachers 3_York, Rural.txt', 'EFO_WomenUnited-11-19-2025 -Karen- FINAL.txt']


,filename,date,location,facilitator,attendees,population_tags,speaker_id,speaker_role,full_text
0,"EFO_AvestaOOB_York County, Older Adults, Rente...",3/17/26,OOB,Pamela Bennett,8,,Speaker 1,unknown,"This is Pamela, and it's March 17, and I am at..."
1,"EFO_AvestaOOB_York County, Older Adults, Rente...",3/17/26,OOB,Pamela Bennett,8,,Speaker 2,unknown,School System. The school system is specific t...
2,"EFO_AvestaOOB_York County, Older Adults, Rente...",3/17/26,OOB,Pamela Bennett,8,,Speaker 3,unknown,I think the police department and fire departm...
3,"EFO_AvestaOOB_York County, Older Adults, Rente...",3/17/26,OOB,Pamela Bennett,8,,Speaker 4,unknown,"No Everybody minds their own business, but the..."
4,"EFO_AvestaOOB_York County, Older Adults, Rente...",3/17/26,OOB,Pamela Bennett,8,,Speaker 6,unknown,"There's a lot of places to go walking, if you ..."


## 2. LLM Role Detection — Facilitator vs. Participant
One LLM call per speaker to determine whether they are the session facilitator
or a community member participant.

Facilitators are **fully removed** before any classification or embedding —
their summarizing language would otherwise contaminate participant clusters.

Results are saved to `df_speakers.csv` as a checkpoint.

In [6]:
ROLE_DETECTION_PROMPT = """\
This is a transcript excerpt from a community focus group in Southern Maine.
Determine whether this speaker is the SESSION FACILITATOR or a COMMUNITY PARTICIPANT.

Signals of a FACILITATOR:
- Asks structured questions to the group
- Transitions between topics ("let's move on to...", "the next question is...")
- Summarizes or reflects back what others said
- Mentions United Way or the organizing body
- Introduces the session or wraps it up
- Thanks participants or explains the process

Signals of a PARTICIPANT:
- Shares personal experiences or opinions
- Raises personal concerns about housing, food, health, costs etc.
- Responds to questions with their own views
- Talks about their own life situation

SPEAKER TEXT:
{full_text}

Return ONLY valid JSON, no explanation, no markdown:
{{"role": "facilitator"}}
or
{{"role": "participant"}}
"""


def detect_role(row):
    """Single LLM call to classify a speaker as facilitator or participant.

    Returns 'facilitator', 'participant', or 'unknown' on failure.
    """
    raw = call_ollama_with_retry(
        prompt=ROLE_DETECTION_PROMPT.format(full_text=row["full_text"]),
        temperature=0,   # deterministic — clear binary question
    )
    if raw is None:
        return "unknown"
    try:
        parsed = json.loads(raw)
        assert parsed["role"] in ["facilitator", "participant"]
        return parsed["role"]
    except Exception:
        return "unknown"


def run_role_detection(df, delay=0.3):
    """Detect facilitator vs. participant for every speaker row.

    'unknown' means the LLM failed — these rows are treated as participants
    conservatively (better to include an ambiguous speaker than drop a real one).
    """
    roles = []
    for i, row in df.iterrows():
        role = detect_role(row)
        roles.append(role)
        print(f"[{i}] {row['filename']} | {row['speaker_id']} → {role}")
        time.sleep(delay)

    df              = df.copy()
    df["speaker_role"] = roles
    return df


df_speakers = run_role_detection(df_speakers)
df_speakers.to_csv("df_speakers.csv", index=False)   # checkpoint

print(f"\nFacilitators detected : {(df_speakers['speaker_role'] == 'facilitator').sum()}")
print(f"Participants detected  : {(df_speakers['speaker_role'] == 'participant').sum()}")
print(f"Unknown (kept)         : {(df_speakers['speaker_role'] == 'unknown').sum()}")
print("\nFacilitators will be fully removed before classification and embedding.")
df_speakers.groupby(["filename", "speaker_id", "speaker_role"]).size().reset_index(name="utterances").head(20)

[0] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 1 → facilitator
[1] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 2 → participant
[2] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 3 → participant
[3] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 4 → participant
[4] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 6 → participant
[5] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 8 → participant
[6] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 5 → participant
[7] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 7 → participant
[8] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 10 → participant
[9] EFO_Bartlett Woods_York County, Older Adults, Urban, ALICE.txt | Speaker 1 → facilitator
[10] EFO_Bartlett Woods_York County, Older Adults, Urban, ALICE.txt | Speaker 2 → participant
[11] EFO_Bartlett Woods_York County, Older Adults, Urban, ALICE.t

,filename,speaker_id,speaker_role,utterances
0,"EFO_AvestaOOB_York County, Older Adults, Rente...",Speaker 1,facilitator,1
1,"EFO_AvestaOOB_York County, Older Adults, Rente...",Speaker 10,participant,1
2,"EFO_AvestaOOB_York County, Older Adults, Rente...",Speaker 2,participant,1
3,"EFO_AvestaOOB_York County, Older Adults, Rente...",Speaker 3,participant,1
4,"EFO_AvestaOOB_York County, Older Adults, Rente...",Speaker 4,participant,1
5,"EFO_AvestaOOB_York County, Older Adults, Rente...",Speaker 5,participant,1
6,"EFO_AvestaOOB_York County, Older Adults, Rente...",Speaker 6,participant,1
7,"EFO_AvestaOOB_York County, Older Adults, Rente...",Speaker 7,participant,1
8,"EFO_AvestaOOB_York County, Older Adults, Rente...",Speaker 8,participant,1
9,"EFO_Bartlett Woods_York County, Older Adults, ...",Speaker 1,facilitator,1


## 3. LLM Classification — 3-Vote Consensus
Each **participant** speaker is classified 3 times independently.  
- **All 3 agree** → auto-accepted  
- **Any disagreement** → flagged for human review in Cell 4

Facilitator rows are fully skipped — they have already been removed at this stage.

Two possible labels: `ALICE` (covers ALICE + ALICE(Poverty)) and `Above ALICE`.  
A third option `Cannot Determine` is available when there is insufficient signal.

In [7]:
CLASSIFICATION_PROMPT = """\
You are analyzing a transcript excerpt from a community focus group in Southern Maine.
Classify the speaker's socioeconomic status based ONLY on signals from their own
personal situation — not their general observations about others.

DEFINITIONS:
- ALICE: Asset Limited, Income Constrained. Includes people below the poverty line
  and working people who struggle to afford basic needs (housing, food, healthcare,
  transportation). Often on fixed income, Social Security, or cutting back on
  necessities.
- Above ALICE: Financially stable. Basic needs are met without significant struggle.
- Cannot Determine: The text does not contain enough personal financial signal
  to classify reliably.

SESSION CONTEXT:
Location: {location}
Population tags: {population_tags}
Date: {date}

SPEAKER TEXT:
{full_text}

Return ONLY valid JSON, no explanation, no markdown:
{{\"alice_class\": \"ALICE\"}}
"""


def classify_once(prompt):
    """Single classification call. Returns label string or None on failure.

    Uses call_ollama_with_retry defined in Cell 0.
    """
    raw = call_ollama_with_retry(prompt, temperature=0.2)
    if raw is None:
        return None
    try:
        parsed = json.loads(raw)
        assert parsed["alice_class"] in ["ALICE", "Above ALICE", "Cannot Determine"]
        return parsed["alice_class"]
    except Exception:
        return None


def classify_speaker(row, n_votes=N_VOTES):
    """Run n_votes classification calls and check for consensus.
    
    Returns:
        alice_class   : agreed label if unanimous, else None
        review_status : 'auto' | 'needs_review' | 'error'
        votes         : list of all raw vote results
    """
    prompt      = CLASSIFICATION_PROMPT.format(
        location=row["location"],
        population_tags=row["population_tags"],
        date=row["date"],
        full_text=row["full_text"],
    )
    votes       = [classify_once(prompt) for _ in range(n_votes)]
    valid_votes = [v for v in votes if v is not None]

    if len(valid_votes) == 0:
        return None, "error", votes

    top_label, top_count = Counter(valid_votes).most_common(1)[0]

    if top_count == n_votes:          # unanimous
        return top_label, "auto", votes
    else:                             # any disagreement
        return None, "needs_review", votes


def run_classification(df, delay=0.3):
    """Run 3-vote classification on all participant rows."""
    alice_classes, review_statuses, all_votes = [], [], []

    for i, row in df.iterrows():
        if row["speaker_role"] == "facilitator":
            alice_classes.append(None)
            review_statuses.append("facilitator")
            all_votes.append([])
            continue

        alice_class, status, votes = classify_speaker(row)
        alice_classes.append(alice_class)
        review_statuses.append(status)
        all_votes.append(votes)

        print(
            f"[{i}] {row['filename']} | {row['speaker_id']} | "
            f"votes={votes} → {status}"
            + (f" → {alice_class}" if alice_class else " ⚑ FLAGGED")
        )
        time.sleep(delay)

    df                    = df.copy()
    df["alice_class"]     = alice_classes
    df["review_status"]   = review_statuses
    df["votes"]           = all_votes
    return df


df_classified = run_classification(df_speakers)
df_classified.to_csv("df_classified.csv", index=False)   # checkpoint

print(f"\nAuto-accepted : {(df_classified['review_status'] == 'auto').sum()}")
print(f"Needs review  : {(df_classified['review_status'] == 'needs_review').sum()}")
print(f"Errors        : {(df_classified['review_status'] == 'error').sum()}")

[1] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 2 | votes=['Cannot Determine', 'Cannot Determine', 'Cannot Determine'] → auto → Cannot Determine
[2] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 3 | votes=['Cannot Determine', 'Cannot Determine', 'Cannot Determine'] → auto → Cannot Determine
[3] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 4 | votes=['ALICE', 'ALICE', 'ALICE'] → auto → ALICE
[4] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 6 | votes=['Cannot Determine', 'Cannot Determine', 'Cannot Determine'] → auto → Cannot Determine
[5] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 8 | votes=['Cannot Determine', 'Cannot Determine', 'Cannot Determine'] → auto → Cannot Determine
[6] EFO_AvestaOOB_York County, Older Adults, Renters.txt | Speaker 5 | votes=['Cannot Determine', 'Cannot Determine', 'Cannot Determine'] → auto → Cannot Determine
[7] EFO_AvestaOOB_York County, Older Adults, Renters.txt | S

## 4. Human Review of Flagged Rows
Only rows where the 3 votes disagreed are shown.  
Read the speaker text and votes, then enter the correct label.  

**Skip this cell if there are no flagged rows.**

In [ ]:
"""VALID_LABELS = ["ALICE", "Above ALICE", "Cannot Determine"]

flagged = df_classified[df_classified["review_status"] == "needs_review"]

if len(flagged) == 0:
    print("No flagged rows — proceed to Cell 4.")
else:
    print(f"{len(flagged)} rows need review.\n")
    for i, row in flagged.iterrows():
        print(f"{'─'*60}")
        print(f"File    : {row['filename']}")
        print(f"Speaker : {row['speaker_id']}")
        print(f"Votes   : {row['votes']}")
        print(f"Text    :\n{row['full_text'][:600]}")
        print()
        while True:
            label = input(f"Label {VALID_LABELS}: ").strip()
            if label in VALID_LABELS:
                df_classified.at[i, "alice_class"]   = label
                df_classified.at[i, "review_status"] = "manual"
                break
            print("Invalid — please enter one of the options above.")

    df_classified.to_csv("df_classified.csv", index=False)   # save after review
    print("\nReview complete. df_classified saved.")"""

10 rows need review.

────────────────────────────────────────────────────────────
File    : EFO_Bartlett Woods_York County, Older Adults, Urban, ALICE.txt
Speaker : Speaker 7
Votes   : ['Cannot Determine', 'Above ALICE', 'Cannot Determine']
Text    :
I would add transportation issues probably coming up, right. So, I mean, I have a car. So, I mean, at this point I can get around, but that's going to become more and more difficult to the point where, you know, do I buy a car? Do I just jettison the car? But there's not a lot of transportation options. I think that's a very good point. I mean, there is no real public transportation. I think I'm the newest member here. Yeah, I moved here in October. I find the community very welcoming. Everyone has been so nice. Yes, I mean, there's 28 units, and there's going to be 28 at least 28 different pe



## 5. LLM Concern Extraction
One call per participant speaker.  
Returns a list of concern summaries — each a single focused sentence.  

Runs independently from classification — concern text is not affected by ALICE label.

In [ ]:
CONCERN_PROMPT = """\
You are analyzing a transcript excerpt from a community focus group in Southern Maine.
Extract every distinct concern this speaker personally expresses.

Rules:
- Each concern must be one clear, specific sentence describing a personal struggle
  or problem the speaker faces
- Focus on what they personally experience — not general observations about others
- Ignore conversational filler, tangents, and noise
- If the speaker expresses no personal concerns, return an empty list

SPEAKER TEXT:
{full_text}

Return ONLY valid JSON, no explanation, no markdown:
{{"concerns": [
  "Cannot afford meat and has switched to cheaper proteins",
  "Medication costs force a tradeoff with food budget"
]}}
"""


def extract_concerns(row):
    """Single LLM call to extract concern summaries from speaker text.
    
    Uses call_ollama_with_retry for robustness against transient Ollama failures.
    """
    raw = call_ollama_with_retry(
        prompt=CONCERN_PROMPT.format(full_text=row["full_text"]),
        temperature=0,   # deterministic — one call is enough
    )
    if raw is None:
        return []
    try:
        parsed = json.loads(raw)
        assert isinstance(parsed["concerns"], list)
        return parsed["concerns"]
    except Exception:
        return []


def run_concern_extraction(df, delay=0.3):
    """Extract concerns for all participant rows with accepted ALICE labels."""
    concerns_list = []
    eligible = df[
        (df["speaker_role"] == "participant") &
        (df["alice_class"].notna()) &
        (~df["alice_class"].isin(["Cannot Determine", "ERROR"])) &
        (df["review_status"].isin(["auto", "manual"]))
    ].index

    for i, row in df.iterrows():
        if i not in eligible:
            concerns_list.append([])
            continue
        concerns = extract_concerns(row)
        concerns_list.append(concerns)
        print(f"[{i}] {row['filename']} | {row['speaker_id']} → {len(concerns)} concerns")
        time.sleep(delay)

    df            = df.copy()
    df["concerns"] = concerns_list
    return df


df_classified = run_concern_extraction(df_classified)
df_classified.to_csv("df_classified.csv", index=False)   # update checkpoint
print("\nConcern extraction complete.")

## 6. Explode Concerns → One Row Per Concern → One Row Per Concern
Each speaker's concern list becomes individual rows.  
Speaker metadata and ALICE class carry over to every concern row.

In [ ]:
def explode_concerns(df):
    """Expand concern lists into one row per concern."""
    eligible = df[
        (df["speaker_role"] == "participant") &
        (df["alice_class"].notna()) &
        (~df["alice_class"].isin(["Cannot Determine", "ERROR"])) &
        (df["review_status"].isin(["auto", "manual"]))
    ].copy()

    df_exploded = (
        eligible
        .explode("concerns")
        .rename(columns={"concerns": "concern"})
    )
    df_exploded = df_exploded[
        df_exploded["concern"].notna() &
        (df_exploded["concern"].str.strip() != "")
    ].reset_index(drop=True)

    print(f"Total concern rows : {len(df_exploded)}")
    print(f"ALICE rows         : {(df_exploded['alice_class'] == 'ALICE').sum()}")
    print(f"Above ALICE rows   : {(df_exploded['alice_class'] == 'Above ALICE').sum()}")
    return df_exploded


df_concerns = explode_concerns(df_classified)
df_concerns[["filename", "speaker_id", "alice_class", "concern"]].head(10)

## 7. Filter to ALICE Speakers
Split into two dataframes for independent clustering.

In [ ]:
def split_by_alice_class(df_concerns):
    """Split into ALICE and Above ALICE concern dataframes."""
    df_alice       = df_concerns[df_concerns["alice_class"] == "ALICE"].reset_index(drop=True)
    df_above_alice = df_concerns[df_concerns["alice_class"] == "Above ALICE"].reset_index(drop=True)

    print(f"ALICE concern rows       : {len(df_alice)}")
    print(f"Above ALICE concern rows : {len(df_above_alice)}")
    return df_alice, df_above_alice


df_alice, df_above_alice = split_by_alice_class(df_concerns)

# Save Above ALICE for later — same pipeline applies
df_above_alice.to_csv("df_above_alice.csv", index=False)
print("df_above_alice saved for later run.")

## 8. Embed Concerns
Use `nomic-embed-text` via Ollama — one call per concern row.  
Embeddings are saved to disk so this step can be skipped on reruns.

In [ ]:
EMBEDDINGS_PATH = "embeddings_alice.npy"


def embed_concerns(df, save_path=EMBEDDINGS_PATH):
    """Embed concern column using nomic-embed-text via Ollama.
    
    - Loads completed embeddings from disk if the file already exists.
    - Saves a checkpoint every 50 rows so a crash mid-run doesn't
      lose all progress — restart will resume from the checkpoint.
    """
    checkpoint_path = save_path.replace(".npy", "_checkpoint.npy")

    # ── Full file already exists — load and return ────────────────────────────
    if Path(save_path).exists():
        print(f"Loading existing embeddings from {save_path}")
        embeddings = np.load(save_path)
        assert len(embeddings) == len(df), \
            "Saved embeddings length mismatch — delete the .npy file and re-run."
        return embeddings

    # ── Partial checkpoint exists — resume from where we left off ─────────────
    if Path(checkpoint_path).exists():
        embeddings = list(np.load(checkpoint_path))
        start_idx  = len(embeddings)
        print(f"Resuming from checkpoint at row {start_idx}/{len(df)}")
    else:
        embeddings = []
        start_idx  = 0
        print(f"Embedding {len(df)} concerns with {EMBED_MODEL}...")

    # ── Embed remaining rows ──────────────────────────────────────────────────
    concerns = df["concern"].tolist()
    for i, concern in enumerate(concerns[start_idx:], start=start_idx):
        response = ollama.embeddings(model=EMBED_MODEL, prompt=concern)
        embeddings.append(response["embedding"])

        # Save checkpoint every 50 rows
        if (i + 1) % 50 == 0:
            np.save(checkpoint_path, np.array(embeddings))
            print(f"  Checkpoint saved at row {i + 1}/{len(df)}")

    # ── Finalise — save full file, remove checkpoint ──────────────────────────
    embeddings = np.array(embeddings)
    np.save(save_path, embeddings)
    if Path(checkpoint_path).exists():
        Path(checkpoint_path).unlink()   # clean up checkpoint
    print(f"Embeddings saved to {save_path} — shape: {embeddings.shape}")
    return embeddings


embeddings_alice = embed_concerns(df_alice)
print(f"\nEmbedding dimensions: {embeddings_alice.shape[1]}")

## 9. UMAP Dimensionality Reduction
Reduce high-dimensional embeddings before clustering.  
Two reductions: 5D for HDBSCAN input, 2D for visualization.

In [ ]:
def reduce_dimensions(embeddings):
    """Run UMAP twice — 5D for clustering, 2D for visualization."""
    print("Reducing to 5D for clustering...")
    reducer_5d = umap.UMAP(
        n_components=UMAP_N_COMPONENTS,
        metric=UMAP_METRIC,
        random_state=UMAP_RANDOM_STATE,
        n_neighbors=min(15, len(embeddings) - 1),   # guard for small datasets
    )
    embeddings_5d = reducer_5d.fit_transform(embeddings)

    print("Reducing to 2D for visualization...")
    reducer_2d = umap.UMAP(
        n_components=UMAP_N_COMPONENTS_2D,
        metric=UMAP_METRIC,
        random_state=UMAP_RANDOM_STATE,
        n_neighbors=min(15, len(embeddings) - 1),
    )
    embeddings_2d = reducer_2d.fit_transform(embeddings)

    print(f"5D shape: {embeddings_5d.shape}")
    print(f"2D shape: {embeddings_2d.shape}")
    return embeddings_5d, embeddings_2d


embeddings_5d, embeddings_2d = reduce_dimensions(embeddings_alice)

## 10. HDBSCAN Clustering
Cluster the 5D UMAP embeddings.  
Points that don't fit any cluster are labeled `-1` (noise).

In [ ]:
def cluster_concerns(df, embeddings_5d):
    """Run HDBSCAN and attach cluster labels to the dataframe.
    
    min_cluster_size is computed dynamically from group size so that
    smaller groups (e.g. Above ALICE) don't produce only noise.
    HDBSCAN_MIN_CLUSTER_SIZE in Cell 0 acts as a floor.
    """
    # Dynamic sizing: ~5% of group size, with a minimum floor from config
    dynamic_min_cluster_size = max(HDBSCAN_MIN_CLUSTER_SIZE, len(df) // 20)
    print(f"Group size: {len(df)} → min_cluster_size: {dynamic_min_cluster_size}")

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=dynamic_min_cluster_size,
        min_samples=HDBSCAN_MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method="eom",
    )
    labels = clusterer.fit_predict(embeddings_5d)

    df           = df.copy()
    df["cluster"] = labels

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()
    print(f"Clusters found : {n_clusters}")
    print(f"Noise points   : {n_noise} ({100 * n_noise / len(labels):.1f}%)")
    print(f"\nCluster sizes:")
    print(pd.Series(labels).value_counts().sort_index().to_string())
    return df


df_alice = cluster_concerns(df_alice, embeddings_5d)
df_alice[["concern", "cluster"]].head(10)

## 11. LLM Cluster Naming + Word Cloud Sanity Check + Word Cloud Sanity Check
For each cluster:
- **LLM generates a short descriptive label** based on the concern list
- **Word cloud visualizes raw term frequency** — use this to verify the LLM label is accurate
- **Sample concerns** shown as ground truth

If the word cloud contradicts the LLM label, trust the word cloud.

In [ ]:
CLUSTER_NAME_PROMPT = """\
Below are concern statements from members of a community focus group,
all belonging to the same thematic cluster.

CONCERNS:
{concerns}

Give a short, specific 4-7 word label that captures the shared theme.
Return ONLY valid JSON, no explanation:
{{"label": "Housing cost burden on fixed income"}}
"""


def name_cluster(concerns):
    """Ask LLM to generate a short descriptive label for a cluster."""
    try:
        prompt   = CLUSTER_NAME_PROMPT.format(
            concerns="\n".join(f"- {c}" for c in concerns)
        )
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            format="json",
            options={"temperature": 0},
        )
        raw    = response["message"]["content"].strip()
        raw    = re.sub(r"^```json\s*|^```\s*|\s*```$", "", raw, flags=re.MULTILINE).strip()
        parsed = json.loads(raw)
        return parsed["label"]
    except Exception:
        return "(label failed)"


def plot_cluster(cluster_id, concerns, llm_label, rank, total):
    """Plot word cloud + sample concerns side by side for one cluster."""
    combined_text = " ".join(concerns)
    wc = WordCloud(
        width=600, height=300,
        background_color="white",
        stopwords=None,         # keep all words — stopwords would hide signal
        max_words=40,
        colormap="Blues",
    ).generate(combined_text)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(
        f"Cluster {rank}/{total}  (id={cluster_id}, n={len(concerns)})\n"
        f"LLM Label: \"{llm_label}\"",
        fontsize=13, fontweight="bold", y=1.02
    )

    # Word cloud
    axes[0].imshow(wc, interpolation="bilinear")
    axes[0].axis("off")
    axes[0].set_title("Word Cloud (sanity check)", fontsize=11)

    # Sample concerns
    axes[1].axis("off")
    axes[1].set_title("Sample Concerns", fontsize=11)
    sample_text = "\n\n".join(f"• {c}" for c in concerns[:5])
    axes[1].text(
        0.02, 0.95, sample_text,
        transform=axes[1].transAxes,
        fontsize=9, verticalalignment="top",
        wrap=True
    )

    plt.tight_layout()
    plt.savefig(f"cluster_{cluster_id}_alice.png", bbox_inches="tight", dpi=150)
    plt.show()


def build_and_display_clusters(df, group_label="ALICE"):
    """Name all clusters, display word cloud + samples, return summary dict."""
    df_clustered = df[df["cluster"] != -1]
    cluster_ids  = sorted(df_clustered["cluster"].unique())

    # Sort by cluster size descending
    cluster_ids = sorted(
        cluster_ids,
        key=lambda c: -len(df_clustered[df_clustered["cluster"] == c])
    )

    summary = {}
    print(f"{'='*60}")
    print(f"  CLUSTER RESULTS — {group_label}")
    print(f"{'='*60}\n")

    for rank, cluster_id in enumerate(cluster_ids, 1):
        concerns   = df_clustered[df_clustered["cluster"] == cluster_id]["concern"].tolist()
        llm_label  = name_cluster(concerns)
        summary[cluster_id] = {"label": llm_label, "size": len(concerns), "concerns": concerns}
        print(f"Cluster {rank}: \"{llm_label}\"  (n={len(concerns)})")
        plot_cluster(cluster_id, concerns, llm_label, rank, len(cluster_ids))

    noise_n = (df["cluster"] == -1).sum()
    print(f"\nNoise (unclustered): {noise_n} concerns")
    return summary


alice_cluster_summary = build_and_display_clusters(df_alice, group_label="ALICE")

## 12. 2D UMAP Visualization
Scatter plot of all ALICE concern rows colored by cluster.  
Noise points (`-1`) shown in grey.  
Cluster centroids labeled with LLM-generated names.

In [ ]:
def plot_umap_clusters(df, embeddings_2d, cluster_summary, group_label="ALICE"):
    """2D UMAP scatter plot with cluster labels at centroids."""
    labels      = df["cluster"].values
    unique_ids  = sorted(set(labels))
    n_clusters  = len([c for c in unique_ids if c != -1])

    # Color palette — one color per cluster, grey for noise
    palette     = cm.get_cmap("tab20", max(n_clusters, 1))
    color_map   = {c: palette(i) for i, c in enumerate(c for c in unique_ids if c != -1)}
    color_map[-1] = (0.7, 0.7, 0.7, 0.4)   # grey, semi-transparent for noise

    fig, ax = plt.subplots(figsize=(12, 8))

    for cluster_id in unique_ids:
        mask   = labels == cluster_id
        points = embeddings_2d[mask]
        color  = color_map[cluster_id]
        label  = "Noise" if cluster_id == -1 else cluster_summary.get(cluster_id, {}).get("label", str(cluster_id))

        ax.scatter(
            points[:, 0], points[:, 1],
            c=[color], s=60, alpha=0.7,
            label=f"{label} (n={mask.sum()})" if cluster_id != -1 else f"Noise (n={mask.sum()})",
            edgecolors="white", linewidths=0.3,
        )

        # Label centroid for non-noise clusters
        if cluster_id != -1:
            cx, cy = points[:, 0].mean(), points[:, 1].mean()
            ax.annotate(
                label, (cx, cy),
                fontsize=8, fontweight="bold",
                ha="center", va="bottom",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7, ec="none")
            )

    ax.set_title(f"UMAP Cluster Map — {group_label} Concerns", fontsize=14, fontweight="bold")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.8)
    plt.tight_layout()
    plt.savefig(f"umap_{group_label.lower().replace(' ', '_')}.png", dpi=150, bbox_inches="tight")
    plt.show()


plot_umap_clusters(df_alice, embeddings_2d, alice_cluster_summary, group_label="ALICE")

## 13. Save ALICE Outputs

In [ ]:
df_alice.drop(columns=["embedding"], errors="ignore").to_csv("df_alice_clustered.csv", index=False)

summary_serializable = {
    str(k): {"label": v["label"], "size": v["size"], "concerns": v["concerns"]}
    for k, v in alice_cluster_summary.items()
}
with open("alice_cluster_summary.json", "w") as f:
    json.dump(summary_serializable, f, indent=2)

print("Saved:")
print("  df_alice_clustered.csv     — ALICE concern rows with cluster labels")
print("  embeddings_alice.npy       — ALICE embeddings")
print("  alice_cluster_summary.json — LLM labels + concerns per cluster")
print("  cluster_*_alice.png        — word cloud + sample plots per cluster")
print("  umap_alice.png             — 2D cluster map")

## 14. Run Above ALICE Through the Same Pipeline Through the Same Pipeline
Reuses all functions from cells 7–11 on `df_above_alice`.  
Embeddings saved separately so neither group's cache is overwritten.

In [ ]:
# ── Reload guard — safe to re-run Cell 13 after a kernel restart ─────────────
if "df_above_alice" not in dir() or len(df_above_alice) == 0:
    print("df_above_alice not found in memory — loading from df_above_alice.csv")
    df_above_alice = pd.read_csv("df_above_alice.csv")
    # concerns column was saved as a string — convert back to list
    df_above_alice["concerns"] = df_above_alice["concerns"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
print(f"Above ALICE rows: {len(df_above_alice)}")

# ── Embed ─────────────────────────────────────────────────────────────────────
embeddings_above = embed_concerns(df_above_alice, save_path="embeddings_above_alice.npy")

# ── UMAP ──────────────────────────────────────────────────────────────────────
embeddings_above_5d, embeddings_above_2d = reduce_dimensions(embeddings_above)

# ── HDBSCAN ───────────────────────────────────────────────────────────────────
df_above_alice = cluster_concerns(df_above_alice, embeddings_above_5d)

# ── LLM cluster naming + word clouds ─────────────────────────────────────────
above_alice_cluster_summary = build_and_display_clusters(
    df_above_alice, group_label="Above ALICE"
)

# ── 2D UMAP plot ──────────────────────────────────────────────────────────────
plot_umap_clusters(
    df_above_alice, embeddings_above_2d,
    above_alice_cluster_summary, group_label="Above ALICE"
)

# ── Save ──────────────────────────────────────────────────────────────────────
df_above_alice.drop(columns=["embedding"], errors="ignore").to_csv(
    "df_above_alice_clustered.csv", index=False
)
above_summary_serializable = {
    str(k): {"label": v["label"], "size": v["size"], "concerns": v["concerns"]}
    for k, v in above_alice_cluster_summary.items()
}
with open("above_alice_cluster_summary.json", "w") as f:
    json.dump(above_summary_serializable, f, indent=2)

print("Above ALICE pipeline complete.")

## 15. Side-by-Side Comparison: ALICE vs. Above ALICE: ALICE vs. Above ALICE
Three outputs:
- **Ranked concern table** for each group — sorted by number of unique speakers, not just raw concern count
- **Shared concerns** — themes that appear prominently in both groups
- **Unique concerns** — what only one group raises

In [ ]:
def count_unique_speakers(df, cluster_id):
    """Count distinct speakers who contributed to a cluster."""
    return df[df["cluster"] == cluster_id]["speaker_id"].nunique()


def build_ranked_table(df, cluster_summary, group_label):
    """Build a ranked dataframe of clusters with concern count + unique speaker count."""
    rows = []
    for cluster_id, info in cluster_summary.items():
        n_speakers = count_unique_speakers(df, cluster_id)
        rows.append({
            "rank":            0,           # filled after sorting
            "cluster_label":   info["label"],
            "n_concerns":      info["size"],
            "n_speakers":      n_speakers,
            "example_concern": info["concerns"][0] if info["concerns"] else "",
        })

    ranked = (
        pd.DataFrame(rows)
        .sort_values("n_speakers", ascending=False)
        .reset_index(drop=True)
    )
    ranked["rank"] = ranked.index + 1
    ranked["group"] = group_label
    return ranked


def _deduplicated_pairs(sim_matrix, alice_labels, above_labels, threshold):
    """Helper: find matching cluster pairs above similarity threshold.
    
    Uses a seen set to prevent duplicate pairs when multiple ALICE clusters
    match the same Above ALICE cluster (or vice versa).
    """
    shared_alice, shared_above = set(), set()
    shared_pairs = []
    seen = set()

    for i, alice_label in enumerate(alice_labels):
        for j, above_label in enumerate(above_labels):
            if sim_matrix[i, j] >= threshold and (i, j) not in seen:
                seen.add((i, j))
                shared_alice.add(i)
                shared_above.add(j)
                shared_pairs.append((alice_label, above_label, round(sim_matrix[i, j], 2)))

    return shared_pairs, shared_alice, shared_above


def find_shared_and_unique(alice_table, above_table, similarity_threshold=0.6):
    """Identify shared vs. unique themes by embedding cluster labels and
    comparing cosine similarity.
    
    Clusters with similarity above threshold are considered shared concerns.
    Deduplication handled by _deduplicated_pairs.
    """
    from sklearn.metrics.pairwise import cosine_similarity as cos_sim

    alice_labels = alice_table["cluster_label"].tolist()
    above_labels = above_table["cluster_label"].tolist()
    all_labels   = alice_labels + above_labels

    # Embed all labels in one batch
    label_embeddings = np.array([
        ollama.embeddings(model=EMBED_MODEL, prompt=label)["embedding"]
        for label in all_labels
    ])

    alice_embs = label_embeddings[:len(alice_labels)]
    above_embs = label_embeddings[len(alice_labels):]
    sim_matrix = cos_sim(alice_embs, above_embs)   # shape: (n_alice, n_above)

    shared_pairs, shared_alice, shared_above = _deduplicated_pairs(
        sim_matrix, alice_labels, above_labels, similarity_threshold
    )

    unique_alice = [l for i, l in enumerate(alice_labels) if i not in shared_alice]
    unique_above = [l for j, l in enumerate(above_labels) if j not in shared_above]

    return shared_pairs, unique_alice, unique_above


def print_comparison(alice_table, above_table, shared_pairs, unique_alice, unique_above):
    """Print a clean side-by-side comparison summary."""
    max_rows = max(len(alice_table), len(above_table))

    print("=" * 100)
    print(f"  CONCERN COMPARISON: ALICE vs. ABOVE ALICE")
    print(f"  Ranked by number of unique speakers raising each concern")
    print("=" * 100)
    print(f"{'Rank':<5} {'ALICE Concerns':<45} {'Spkrs':>5}   {'Above ALICE Concerns':<45} {'Spkrs':>5}")
    print("-" * 100)

    for i in range(max_rows):
        if i < len(alice_table):
            a_label  = alice_table.iloc[i]["cluster_label"][:43]
            a_spkrs  = int(alice_table.iloc[i]["n_speakers"])
        else:
            a_label, a_spkrs = "", ""

        if i < len(above_table):
            b_label  = above_table.iloc[i]["cluster_label"][:43]
            b_spkrs  = int(above_table.iloc[i]["n_speakers"])
        else:
            b_label, b_spkrs = "", ""

        print(f"{i+1:<5} {a_label:<45} {str(a_spkrs):>5}   {b_label:<45} {str(b_spkrs):>5}")

    print()
    print("─" * 100)
    print("SHARED CONCERNS (appear in both groups):")
    if shared_pairs:
        for alice_l, above_l, sim in shared_pairs:
            print(f"  ALICE: '{alice_l}'")
            print(f"  Above: '{above_l}'  (similarity: {sim})")
            print()
    else:
        print("  None found above similarity threshold.")

    print("─" * 100)
    print("UNIQUE TO ALICE:")
    for label in unique_alice:
        print(f"  • {label}")

    print()
    print("UNIQUE TO ABOVE ALICE:")
    for label in unique_above:
        print(f"  • {label}")
    print("=" * 100)


# ── Build ranked tables ───────────────────────────────────────────────────────
alice_ranked = build_ranked_table(df_alice,       alice_cluster_summary,       "ALICE")
above_ranked = build_ranked_table(df_above_alice, above_alice_cluster_summary, "Above ALICE")

# ── Find shared vs. unique ────────────────────────────────────────────────────
shared_pairs, unique_alice, unique_above = find_shared_and_unique(alice_ranked, above_ranked)

# ── Print comparison ──────────────────────────────────────────────────────────
print_comparison(alice_ranked, above_ranked, shared_pairs, unique_alice, unique_above)

# ── Save comparison table ─────────────────────────────────────────────────────
comparison_df = pd.concat([alice_ranked, above_ranked], ignore_index=True)
comparison_df.to_csv("comparison_table.csv", index=False)
print("\ncomparison_table.csv saved.")

## 16. Side-by-Side 2D UMAP Visualization
Both cluster maps in one figure for direct visual comparison.  
Each dot is one concern. Color = cluster. Grey = noise.

In [ ]:
def plot_side_by_side_umap(
    df_alice, embeddings_alice_2d, alice_summary,
    df_above, embeddings_above_2d, above_summary,
):
    """Two UMAP scatter plots side by side for direct comparison."""

    def draw_panel(ax, df, embeddings_2d, cluster_summary, group_label, palette_name):
        labels     = df["cluster"].values
        unique_ids = sorted(set(labels))
        n_clusters = len([c for c in unique_ids if c != -1])
        palette    = cm.get_cmap(palette_name, max(n_clusters, 1))
        color_map  = {c: palette(i) for i, c in enumerate(c for c in unique_ids if c != -1)}
        color_map[-1] = (0.75, 0.75, 0.75, 0.3)

        for cluster_id in unique_ids:
            mask   = labels == cluster_id
            points = embeddings_2d[mask]
            color  = color_map[cluster_id]
            lbl    = (
                f"Noise (n={mask.sum()})"
                if cluster_id == -1
                else cluster_summary.get(cluster_id, {}).get("label", str(cluster_id))
            )
            ax.scatter(
                points[:, 0], points[:, 1],
                c=[color], s=55, alpha=0.75,
                label=lbl if cluster_id != -1 else f"Noise (n={mask.sum()})",
                edgecolors="white", linewidths=0.3,
            )
            if cluster_id != -1 and len(points) > 0:
                cx, cy = points[:, 0].mean(), points[:, 1].mean()
                ax.annotate(
                    lbl, (cx, cy),
                    fontsize=7.5, fontweight="bold", ha="center", va="bottom",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.75, ec="none")
                )

        ax.set_title(f"{group_label} Concerns\n({n_clusters} clusters, {len(df)} total)",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("UMAP 1")
        ax.set_ylabel("UMAP 2")
        ax.legend(loc="lower right", fontsize=7, framealpha=0.8,
                  title="Clusters", title_fontsize=8)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(
        "Concern Cluster Comparison: ALICE vs. Above ALICE",
        fontsize=15, fontweight="bold", y=1.01
    )

    draw_panel(axes[0], df_alice,       embeddings_alice_2d, alice_summary, "ALICE",       "Blues")
    draw_panel(axes[1], df_above_alice, embeddings_above_2d, above_summary, "Above ALICE", "Oranges")

    plt.tight_layout()
    plt.savefig("umap_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("umap_comparison.png saved.")


plot_side_by_side_umap(
    df_alice,       embeddings_2d,       alice_cluster_summary,
    df_above_alice, embeddings_above_2d, above_alice_cluster_summary,
)

# ── Final save summary ────────────────────────────────────────────────────────
print("\nAll outputs saved:")
print("  df_classified.csv              — speaker rows with ALICE labels + concerns")
print("  df_alice_clustered.csv         — ALICE concern rows with cluster labels")
print("  df_above_alice_clustered.csv   — Above ALICE concern rows with cluster labels")
print("  embeddings_alice.npy           — ALICE embeddings cache")
print("  embeddings_above_alice.npy     — Above ALICE embeddings cache")
print("  alice_cluster_summary.json     — ALICE cluster labels + concerns")
print("  above_alice_cluster_summary.json — Above ALICE cluster labels + concerns")
print("  comparison_table.csv           — ranked concern table for both groups")
print("  cluster_*_alice.png            — ALICE word cloud plots")
print("  cluster_*_above_alice.png      — Above ALICE word cloud plots")
print("  umap_alice.png                 — ALICE 2D cluster map")
print("  umap_above_alice.png           — Above ALICE 2D cluster map")
print("  umap_comparison.png            — side-by-side comparison map")